# WattWise — Heuristic (Genetic Algorithm) for Battery Dispatch

Minimize the daily electricity bill by scheduling battery charging and discharging
in each 30-minute slot under CEB's Time-of-Use tariff.

**Method:** Genetic Algorithm (heuristic) — compared with the DP (exact method).

---
> **Shared code.** The data loader, battery parameters and `simulate()` now live in
> `src/model.py`, and the GA (`fitness`, `greedy_schedule`, `run_ga`) in
> `src/heuristic_solver.py`. The DP (`src/dp_solver.py`) imports the same model, so both
> methods optimise and are scored by exactly the same objective. The cells below import
> them; `inspect.getsource` prints the code where it used to be defined.


## Stage 1 — Data Loading

Load the cleaned 30-minute profiles into separate per-day arrays for solar, demand, and price.

### 1.1 Imports and data-loading functions

In [ ]:
# import necessary libraries
import sys
import inspect
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("src").resolve()))  # run from the repo root

In [ ]:
# Data loading is shared with the DP: src/model.py
from model import DATA_PATH, SLOTS_PER_DAY, SLOT_HOURS, load_profiles, list_days, get_day

print(inspect.getsource(get_day))

### 1.2 Sanity check — load one day and inspect it

In [ ]:
# sanity check
df = load_profiles()
print("Days available:", list_days(df))

day = get_day(df, "2026-09-11")
print(f"\nLoaded {day['date']}")
print(f"  slots       : {len(day['solar'])}")
print(f"  total solar : {day['solar'].sum():.2f} kWh")
print(f"  total demand: {day['demand'].sum():.2f} kWh")
print(f"  price levels: {sorted(set(day['price']))} Rs/kWh")

### 1.3 Inspect slots, bands and prices
See how the 48 slots map to tariff bands and prices, and where the bands switch.

In [ ]:
day = get_day(df, "2026-09-11")

# full 48-slot table: index, time, band, price
print("idx  time    band       price")
print("-" * 34)
for i in range(SLOTS_PER_DAY):
    print(f"{i:>3}  {day['slot'][i]:>5}   {day['band'][i]:<9}  {day['price'][i]:>5.0f}")

# where do the bands change? (should land on 05:30, 18:30, 22:30)
print("\nBand transitions:")
prev = None
for i in range(SLOTS_PER_DAY):
    if day["band"][i] != prev:
        print(f"  slot {i:>2} ({day['slot'][i]}) -> {day['band'][i]} (Rs {day['price'][i]:.0f})")
        prev = day["band"][i]

# how many slots in each band
print("\nSlots per band:")
bands, counts = np.unique(day["band"], return_counts=True)
for b, n in zip(bands, counts):
    print(f"  {b:<9}: {n} slots")

---

## Stage 2 — Battery Simulation & Bill Calculation

This is the core of the project: a function that takes a battery schedule (48 actions) and a battery size, simulates the battery throughout the day, and returns the electricity bill. This bill **is the objective function** that the GA will minimize.

**Sign convention:** positive action = discharge, negative = charge.


### 2.1 Battery constants (shared with the DP — keep identical)

In [ ]:
# Battery model parameters, shared with the DP: src/model.py
from model import (CAPACITIES, SOC_RESERVE_FRAC, CHARGE_RATE_KW, DISCHARGE_RATE_KW,
                   MAX_CHARGE_KWH, MAX_DISCHARGE_KWH, ETA_C, ETA_D)
from heuristic_solver import PENALTY

print(f"reserve {SOC_RESERVE_FRAC}, charge <= {MAX_CHARGE_KWH} kWh/slot, "
      f"discharge <= {MAX_DISCHARGE_KWH} kWh/slot, eta_c = eta_d = {ETA_C} "
      f"(calibrated from the logs, scripts/calibrate_eta.py), penalty {PENALTY:g}")

### 2.2 The simulation + bill function (the objective)

Converts a battery **schedule** (48 actions) into an **electricity bill** — this is the
objective function the GA minimizes. Walking slot by slot, it tracks the battery level
(SoC), checks the rules (rate, no export, capacity), and sums the grid cost each slot.

**Returns:** `bill` (Rs), `violation` (0 if legal), `feasible` (bool), `soc` (battery level per slot).

In [ ]:
# The simulator is shared with the DP: src/model.py
from model import simulate

print(inspect.getsource(simulate))

**Run the simulation on a do-nothing schedule (should match the no-battery baseline)**

In [ ]:
day = get_day(df, "2026-09-11")

# run the simulation on a do-nothing schedule (should match the no-battery baseline)
result = simulate(np.zeros(SLOTS_PER_DAY), day, capacity=4)

# inspect what the function returns
print("Keys returned:", list(result.keys()))
print(f"  bill      : Rs {result['bill']:.2f}")
print(f"  violation : {result['violation']:.3f}")
print(f"  feasible  : {result['feasible']}")
print(f"  soc       : array of {len(result['soc'])} values (battery level each slot)")
print(f"  soc[:5]   : {result['soc'][:5]}") # there are 48 values

---

## Stage 3 — Constraint Penalties → Fitness Function

The GA minimizes **fitness**, not the raw bill. Fitness is the bill **plus a penalty** for any rule violations:

> **fitness = bill + P × violation**

A feasible schedule (violation = 0) keeps its true bill. An infeasible schedule gets a large penalty added, so the GA avoids it, even if its raw bill looks cheap (for example, by using energy the battery does not have).

This is the **graded penalty** method: the penalty increases with *how badly* a schedule breaks the rules, giving the GA a direction back toward feasibility.


### 3.1 The fitness function

In [ ]:
from heuristic_solver import fitness

print(inspect.getsource(fitness))

### 3.2 Check the penalty makes infeasible schedules lose

In [ ]:
day = get_day(df, "2026-09-11")

def report(name, r):
    print(f"{name:<28} bill={r['bill']:>8.2f}  "
          f"violation={r['violation']:>7.2f}  feasible={str(r['feasible']):>5}  "
          f"fitness={r['fitness']:>10.2f}")

# a feasible saving schedule
good = np.zeros(SLOTS_PER_DAY)
for t in [20, 22, 24, 26, 28]: good[t] = -0.5
for t in range(37, 45): good[t] = +0.12

# an infeasible "cheat" schedule (discharges from an empty battery)
cheat = np.zeros(SLOTS_PER_DAY)
cheat[0] = 1.0
cheat[1] = 1.0

report("do-nothing (feasible)", fitness(np.zeros(SLOTS_PER_DAY), day, 4))
report("sensible saver (feasible)", fitness(good, day, 4))
report("cheat: empty discharge", fitness(cheat, day, 4))

The penalty works. The feasible **sensible saver** (Rs 146.59) beats the Rs 160.91 baseline and keeps its low fitness. The **cheat** looks cheap (Rs 147.02) but discharges energy the battery never had. Its violation of 95 pushes the fitness to around 95,000, so the GA rejects it.


---

## Stage 4 — The Genetic Algorithm (Heuristic Optimizer)

Evolve a population of 48-slot schedules toward the cheapest feasible one. Each generation uses **tournament selection** → **blend crossover** → **Gaussian mutation** → **elitism**.

The GA minimizes the `fitness` from Stage 3 (bill + penalty).


### 4.1 Warm-starting the population with a greedy schedule
Instead of starting the GA from pure random noise, we seed the population with **one** sensible
rule-based schedule (charge surplus solar, discharge at peak). This speeds up convergence and
improves the final answer, while seeding just one keeps the population diverse. It also doubles
as a simple baseline.

In [ ]:
from heuristic_solver import greedy_schedule

print(inspect.getsource(greedy_schedule))

***Test the greedy seed***

Check it produces a feasible schedule and a sensible bill across all battery sizes.

In [ ]:
day = get_day(df, "2026-09-11")
baseline = 160.91

print("Greedy seed check (Sep 11):")
for cap in CAPACITIES:  # [0, 2, 4, 7]
    g = greedy_schedule(day, cap)
    r = simulate(g, day, cap)
    saving = baseline - r["bill"]
    print(f"  cap={cap} kWh :  bill = Rs {r['bill']:6.2f}   "
          f"feasible={r['feasible']}   saving = Rs {saving:5.2f} "
          f"({100*saving/baseline:4.1f}%)")

# show the greedy schedule for one capacity, slot by slot
print("\nGreedy schedule for 4 kWh battery (non-zero slots only):")
print(f"{'slot':>4} {'time':>5} {'band':<9} {'action':>8}  what")

g = greedy_schedule(day, 4)
for t in range(SLOTS_PER_DAY):
    if abs(g[t]) > 1e-9:  # only show slots where something happens
        action = "charge " if g[t] < 0 else "discharge"
        print(f"{t:>4} {day['slot'][t]:>5} {day['band'][t]:<9} {g[t]:>+8.2f}  {action} {abs(g[t]):.2f} kWh")

### 4.2 The Genetic Algorithm
Evolve the population with tournament selection, blend crossover, Gaussian mutation,
and elitism — minimizing the penalized fitness over many generations.
Per-slot bounds (rate limits **and** no-export, `a_t <= max(0, d_t - g_t)`) are enforced by
clipping each gene; the penalty handles the SoC limits, which couple slots together.

In [ ]:
from heuristic_solver import run_ga

print(inspect.getsource(run_ga))

### 4.3 Run the GA on one scenario

A scenario is one (day + battery size) combination. Here, we test the GA on a single scenario, Sep 11 with a 4 kWh battery, to check that it converges and finds a feasible, cheaper schedule before running all 20 scenarios (5 days × 4 sizes) in Stage 5.

In [ ]:
day = get_day(df, "2026-09-11")
baseline = 160.91
result = run_ga(day, capacity=4, seed=1)

print("GA result (Sep 11, 4 kWh battery)")
print(f"  bill      : Rs {result['bill']:.2f}")
print(f"  feasible  : {result['feasible']}")
print(f"  vs no-battery (Rs {baseline}) : saving Rs {baseline - result['bill']:.2f} "
      f"({100*(baseline - result['bill'])/baseline:.1f}%)")
greedy_bill = simulate(greedy_schedule(day, 4), day, 4)["bill"]
print(f"  vs greedy (Rs {greedy_bill:.2f})          : Rs {greedy_bill - result['bill']:.2f} cheaper")
print(f"  convergence : gen0={result['history'][0]:.1f} -> "
      f"gen50={result['history'][49]:.1f} -> gen199={result['history'][-1]:.1f}")

***View the optimized schedule***

In [ ]:
# show the GA's optimized schedule (the solution)
sch = result["schedule"]
soc = result["soc"]

print("GA optimized schedule (Sep 11, 4 kWh) — active slots only:")
print(f"{'slot':>4} {'time':>5} {'band':<9} {'action':>8}  {'what':<18} {'SoC':>5}")

for t in range(SLOTS_PER_DAY):
    if abs(sch[t]) > 0.02:  # skip near-idle slots
        act = "charge" if sch[t] < 0 else "discharge"
        what = f"{act} {abs(sch[t]):.2f} kWh"
        print(f"{t:>4} {day['slot'][t]:>5} {day['band'][t]:<9} "
              f"{sch[t]:>+8.2f}  {what:<18} {soc[t]:>5.2f}")

---
## Stage 5 — Experiments & Plots

Run the GA across the full sweep (4 battery sizes × 5 days = 20 scenarios), each with
multiple random seeds for rigor. Collect the results and visualize them.

### 5.1 Run the full capacity sweep (20 scenarios × seeds)

Run the GA on all 20 scenarios (5 days × 4 sizes), 3 seeds each, and record the bills and savings.

In [ ]:
import time

DAYS = list_days(df)
SEEDS = [1, 2, 3]

rows = []
all_feasible = True          # track across the whole sweep
t0 = time.time()

for date in DAYS:
    day = get_day(df, date)
    baseline = simulate(np.zeros(SLOTS_PER_DAY), day, 0)["bill"]

    for cap in CAPACITIES:
        runs = [run_ga(day, cap, seed=s) for s in SEEDS]   # keep full results
        bills = [r["bill"] for r in runs]
        feas  = [r["feasible"] for r in runs]              # feasibility of each seed

        if not all(feas):
            all_feasible = False
            print(f"⚠️ INFEASIBLE: {date}, cap={cap} kWh — seeds feasible: {feas}")

        rows.append({
            "date": date,
            "capacity": cap,
            "baseline": round(baseline, 1),
            "ga_mean": round(np.mean(bills), 1),
            "ga_best": round(np.min(bills), 1),
            "all_feasible": all(feas),                      # new column
            "saving_pct": round(100 * (baseline - np.min(bills)) / baseline, 1)
                if baseline > 0 else 0.0,
        })

results = pd.DataFrame(rows)

print(f"\nCompleted {len(DAYS)*len(CAPACITIES)*len(SEEDS)} GA runs in {time.time()-t0:.1f}s")
print(f"ALL RUNS FEASIBLE: {all_feasible}\n")      # the headline check
print(results.to_string(index=False))

### 5.2 Summary — average saving by battery size

In [ ]:
summary = results.groupby("capacity").agg(
    avg_bill=("ga_best", "mean"),
    avg_saving_pct=("saving_pct", "mean")
).round(1)

print("Average across the 5 days:")
print(summary.to_string())

### 5.3 Plot 1 — GA convergence

In [ ]:
import matplotlib.pyplot as plt

day = get_day(df, "2026-09-11")
r = run_ga(day, capacity=4, seed=1)

plt.figure(figsize=(7, 4))
plt.plot(r["history"], color="#1a7f5a", lw=1.8, label="GA best fitness")
plt.xlabel("Generation"); plt.ylabel("Best bill (Rs)")
plt.title("GA convergence (Sep 11, 4 kWh)"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 5.4 Plot 2 — Optimized dispatch schedule

In [ ]:
sch, soc = r["schedule"], r["soc"]
x = np.arange(SLOTS_PER_DAY)

fig, ax1 = plt.subplots(figsize=(9, 4))
colors = ["#2d8a5f" if a < 0 else "#c0562a" for a in sch]   # green=charge, red=discharge
ax1.bar(x, sch, color=colors, width=0.8)
ax1.axhline(0, color="#333", lw=0.6)
ax1.axvspan(37, 44.5, color="#ffdede", alpha=0.5, zorder=0, label="Peak band")
ax1.set_xlabel("Slot (0=00:00 ... 47=23:30)")
ax1.set_ylabel("Battery action (kWh)  [− charge / + discharge]")
ax1.set_title("GA optimized dispatch (Sep 11, 4 kWh)")

ax2 = ax1.twinx()
ax2.plot(x, soc, color="#2d5c86", lw=1.8, marker=".", ms=3, label="SoC")
ax2.set_ylabel("State of charge (kWh)", color="#2d5c86")
ax1.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

### 5.5 Plot 3 — Savings vs battery size

In [ ]:
g = results.groupby("capacity")["saving_pct"].mean()
plt.figure(figsize=(6.5, 4))
plt.plot(g.index, g.values, "o-", color="#1a7f5a", lw=2, ms=8)
for cap, val in g.items():
    plt.annotate(f"{val:.0f}%", (cap, val), textcoords="offset points", xytext=(0, 8), fontsize=9)
plt.xlabel("Battery capacity (kWh)"); plt.ylabel("Avg saving vs no-battery (%)")
plt.title("Savings vs battery size (diminishing returns)"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()